# Mistral-7B Abstract Evaluator (Modular)

This notebook orchestrates reusable utilities from:
- `experiments/utils` (model-agnostic)
- `experiments/mistral/utils` (Mistral-specific)

Model used in this notebook:
- `mistralai/Mistral-7B-Instruct-v0.3`

Constraints baked in:
- No quantization (`use_4bit=False`)
- L4 24GB-friendly train/eval batch sizes
- No resume-from-epoch-2 flow


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd


def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "experiments").exists() and (p / "data").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


PosixPath('/home/HaniWB/my_gpu_project/Abstract-Evaluator')

In [2]:
from experiments.mistral.utils.chat import add_messages_and_targets
from experiments.mistral.utils.configs import build_data_paths, build_mistral7_default_config
from experiments.mistral.utils.modeling import load_mistral_model_for_inference
from experiments.mistral.utils.pipeline import (
    estimate_mistral_token_percentiles,
    evaluate_base_model,
    evaluate_saved_adapters,
    evaluate_single_adapter,
)
from experiments.mistral.utils.training import train_mistral7

from experiments.utils.data import (
    clean_train_val_test,
    load_train_val_test_dfs,
    score_distribution,
)
from experiments.utils.datasets_io import export_split_jsonl, to_hf_dataset_dict
from experiments.utils.evaluation import parse_rationale, parse_score
from experiments.utils.generation import generate_predictions_from_messages
from experiments.utils.logging_utils import setup_logger
from experiments.utils.runtime import configure_wandb_dir, set_global_seed


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.7.1+cu118).


In [3]:
if "build_mistral7_default_config" not in globals():
    from experiments.mistral.utils.configs import build_mistral7_default_config

cfg = build_mistral7_default_config(PROJECT_ROOT)

# --- Core run identity ---
cfg.model_name = "mistralai/Mistral-7B-Instruct-v0.3"
cfg.run_name = "mistral7b_abstract_evaluator_lora_l4_modular"

# --- Data paths (edit if you want different splits) ---
cfg.data_paths = build_data_paths(
    train_path=PROJECT_ROOT / "data/data/train/all.jsonl",
    val_path=PROJECT_ROOT / "data/data/val/all.jsonl",
    test_path=PROJECT_ROOT / "data/data/test/all.jsonl",
)

# --- Trainer config (L4 24GB-safe + no quantization) ---
cfg.max_seq_length = 2048
cfg.use_4bit = False
cfg.train.num_train_epochs = 30
cfg.train.per_device_train_batch_size = 2
cfg.train.per_device_eval_batch_size = 4
cfg.train.gradient_accumulation_steps = 4
cfg.train.learning_rate = 8e-5
cfg.train.logging_steps = 5

# Eval/save once per epoch
cfg.train.eval_strategy = "epoch"
cfg.train.eval_steps = None
cfg.train.save_strategy = "epoch"
cfg.train.save_steps = None
cfg.train.save_total_limit = 20

# Stop when validation loss no longer improves
cfg.train.early_stopping_patience = 3
cfg.train.early_stopping_threshold = 0.0

# Throughput/stability settings
cfg.train.dataloader_num_workers = 4
cfg.train.dataloader_pin_memory = True
cfg.train.auto_find_batch_size = True
cfg.train.tf32 = True
cfg.train.gradient_checkpointing = True

# --- Generation/eval config ---
cfg.generation.max_new_tokens = 120
cfg.generation.batch_size = cfg.train.per_device_eval_batch_size

# --- W&B ---
cfg.wandb.enabled = True
cfg.wandb.project = "abstract-evaluator-mistral7b-sft"
cfg.wandb.entity = None
cfg.wandb.tags = ["mistral7b", "lora", "sft", "modular", "l4-24gb", "no-quant"]

cfg.ensure_dirs()
cfg.as_dict()


{'seed': 3407,
 'model_name': 'mistralai/Mistral-7B-Instruct-v0.3',
 'run_name': 'mistral7b_abstract_evaluator_lora_l4_modular',
 'max_seq_length': 2048,
 'use_4bit': False,
 'output_root': '/home/HaniWB/my_gpu_project/Abstract-Evaluator/experiments/artifacts/abstract_evaluator_mistral7b_sft',
 'data_paths': {'train_path': '/home/HaniWB/my_gpu_project/Abstract-Evaluator/data/data/train/all.jsonl',
  'val_path': '/home/HaniWB/my_gpu_project/Abstract-Evaluator/data/data/val/all.jsonl',
  'test_path': '/home/HaniWB/my_gpu_project/Abstract-Evaluator/data/data/test/all.jsonl',
  'combined_path': None},
 'train': {'num_train_epochs': 30,
  'per_device_train_batch_size': 2,
  'per_device_eval_batch_size': 4,
  'gradient_accumulation_steps': 4,
  'learning_rate': 8e-05,
  'warmup_ratio': 0.03,
  'weight_decay': 0.01,
  'lr_scheduler_type': 'cosine',
  'logging_steps': 5,
  'save_total_limit': 20,
  'early_stopping_patience': 3,
  'early_stopping_threshold': 0.0,
  'eval_strategy': 'epoch',
  '

In [4]:
set_global_seed(cfg.seed)
configure_wandb_dir(str(cfg.wandb.dir))

log_dir = cfg.output_root / "logs"
logger = setup_logger(
    name=f"{cfg.run_name}_pipeline",
    log_dir=log_dir,
    log_file=f"{cfg.run_name}.log",
)
logger.info("Initialized run config: %s", json.dumps(cfg.as_dict(), ensure_ascii=False))
log_dir


2026-05-26 20:43:42 | INFO | mistral7b_abstract_evaluator_lora_l4_modular_pipeline | Initialized run config: {"seed": 3407, "model_name": "mistralai/Mistral-7B-Instruct-v0.3", "run_name": "mistral7b_abstract_evaluator_lora_l4_modular", "max_seq_length": 2048, "use_4bit": false, "output_root": "/home/HaniWB/my_gpu_project/Abstract-Evaluator/experiments/artifacts/abstract_evaluator_mistral7b_sft", "data_paths": {"train_path": "/home/HaniWB/my_gpu_project/Abstract-Evaluator/data/data/train/all.jsonl", "val_path": "/home/HaniWB/my_gpu_project/Abstract-Evaluator/data/data/val/all.jsonl", "test_path": "/home/HaniWB/my_gpu_project/Abstract-Evaluator/data/data/test/all.jsonl", "combined_path": null}, "train": {"num_train_epochs": 30, "per_device_train_batch_size": 2, "per_device_eval_batch_size": 4, "gradient_accumulation_steps": 4, "learning_rate": 8e-05, "warmup_ratio": 0.03, "weight_decay": 0.01, "lr_scheduler_type": "cosine", "logging_steps": 5, "save_total_limit": 20, "early_stopping_pati

PosixPath('/home/HaniWB/my_gpu_project/Abstract-Evaluator/experiments/artifacts/abstract_evaluator_mistral7b_sft/logs')

In [5]:
train_df, val_df, test_df = load_train_val_test_dfs(
    train_path=cfg.data_paths.train_path,
    val_path=cfg.data_paths.val_path,
    test_path=cfg.data_paths.test_path,
)

train_df, val_df, test_df = clean_train_val_test(train_df, val_df, test_df)

print("Shapes:", train_df.shape, val_df.shape, test_df.shape)
print("Train score dist:", score_distribution(train_df))
print("Val score dist:", score_distribution(val_df))
print("Test score dist:", score_distribution(test_df))


Shapes: (3055, 42) (298, 42) (298, 42)
Train score dist: {0: 0.10605564648117839, 1: 0.1656301145662848, 2: 0.3509001636661211, 3: 0.22585924713584288, 4: 0.15155482815057283}
Val score dist: {0: 0.10067114093959731, 1: 0.174496644295302, 2: 0.348993288590604, 3: 0.22483221476510068, 4: 0.15100671140939598}
Test score dist: {0: 0.10067114093959731, 1: 0.17114093959731544, 2: 0.348993288590604, 3: 0.22818791946308725, 4: 0.15100671140939598}


In [6]:
train_df = add_messages_and_targets(train_df)
val_df = add_messages_and_targets(val_df)
test_df = add_messages_and_targets(test_df)

print(json.dumps(train_df.iloc[0]["messages"], indent=2, ensure_ascii=False)[:2000])


[
  {
    "role": "system",
    "content": "You are a strict research abstract evaluator. You return only valid JSON."
  },
  {
    "role": "user",
    "content": "Task:\nEvaluate the quality of the following research abstract for conference acceptance.\n\nReference:\nA strong research abstract clearly presents the problem, methodology, contribution, and experimental evidence.\n\nRubric:\nScore scale:\n0 = Very poor abstract: missing most core components, unclear, generic, or unusable.\n1 = Weak abstract: contains a few useful elements but major components are missing or vague.\n2 = Borderline abstract: understandable but incomplete; some important components are weak or missing.\n3 = Good abstract: mostly complete, clear, and logically structured, with minor weaknesses.\n4 = Excellent abstract: complete, clear, concise, well-structured, and strongly communicates the paper's contribution and evidence.\n\nCriteria:\n10_specificity_and_evidence: Avoids generic claims and supports stateme

In [7]:
jsonl_paths = export_split_jsonl(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    output_dir=cfg.output_root / "jsonl",
)

ds = to_hf_dataset_dict(train_df, val_df, test_df)

print("JSONL paths:", jsonl_paths)
print(ds)


JSONL paths: {'train_jsonl': '/home/HaniWB/my_gpu_project/Abstract-Evaluator/experiments/artifacts/abstract_evaluator_mistral7b_sft/jsonl/train.jsonl', 'validation_jsonl': '/home/HaniWB/my_gpu_project/Abstract-Evaluator/experiments/artifacts/abstract_evaluator_mistral7b_sft/jsonl/validation.jsonl', 'test_jsonl': '/home/HaniWB/my_gpu_project/Abstract-Evaluator/experiments/artifacts/abstract_evaluator_mistral7b_sft/jsonl/test.jsonl'}
DatasetDict({
    train: Dataset({
        features: ['id', 'paper_id', 'messages', 'score', 'rationale', 'target_json'],
        num_rows: 3055
    })
    validation: Dataset({
        features: ['id', 'paper_id', 'messages', 'score', 'rationale', 'target_json'],
        num_rows: 298
    })
    test: Dataset({
        features: ['id', 'paper_id', 'messages', 'score', 'rationale', 'target_json'],
        num_rows: 298
    })
})


In [8]:
# Optional: token-length diagnostics (loads base tokenizer/model)
RUN_TOKEN_STATS = False

if RUN_TOKEN_STATS:
    lens = estimate_mistral_token_percentiles(
        model_name=cfg.model_name,
        max_seq_length=cfg.max_seq_length,
        df=pd.concat([train_df, val_df, test_df], ignore_index=True),
    )
    print(lens)


In [9]:
# Main training run (fresh start flow; no resume-from-epoch-2 behavior)
RUN_TRAINING =False

output_dir = cfg.output_root / "models" / cfg.run_name

print({"train_rows": len(train_df), "val_rows": len(val_df), "test_rows": len(test_df)})

if RUN_TRAINING:
    run_info = train_mistral7(
        cfg=cfg,
        ds=ds,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        include_bertscore_for_epoch_eval=False,
        run_epoch_generation_eval=True,
        run_epoch_test_eval=False,
        checkpoint_every_n_epochs=1,
        generation_eval_every_n_epochs=1,
        resume_from_checkpoint=None,
        logger=logger,
    )
else:
    print("RUN_TRAINING=False -> using eval-only mode with existing adapters.")
    run_info = {
        "model_name": cfg.model_name,
        "run_name": cfg.run_name,
        "output_dir": str(output_dir),
        "adapter_dir": str(output_dir / "best_adapter"),
        "epoch_adapter_dir": str(output_dir / "epoch_adapters"),
        "eval_dir": str(cfg.output_root / "eval" / cfg.run_name),
    }

run_info


{'train_rows': 3055, 'val_rows': 298, 'test_rows': 298}
RUN_TRAINING=False -> using eval-only mode with existing adapters.


{'model_name': 'mistralai/Mistral-7B-Instruct-v0.3',
 'run_name': 'mistral7b_abstract_evaluator_lora_l4_modular',
 'output_dir': '/home/HaniWB/my_gpu_project/Abstract-Evaluator/experiments/artifacts/abstract_evaluator_mistral7b_sft/models/mistral7b_abstract_evaluator_lora_l4_modular',
 'adapter_dir': '/home/HaniWB/my_gpu_project/Abstract-Evaluator/experiments/artifacts/abstract_evaluator_mistral7b_sft/models/mistral7b_abstract_evaluator_lora_l4_modular/best_adapter',
 'epoch_adapter_dir': '/home/HaniWB/my_gpu_project/Abstract-Evaluator/experiments/artifacts/abstract_evaluator_mistral7b_sft/models/mistral7b_abstract_evaluator_lora_l4_modular/epoch_adapters',
 'eval_dir': '/home/HaniWB/my_gpu_project/Abstract-Evaluator/experiments/artifacts/abstract_evaluator_mistral7b_sft/eval/mistral7b_abstract_evaluator_lora_l4_modular'}

In [10]:
# Optional: evaluate base model (no finetuning) with BERTScore
RUN_BASE_EVAL = False

if RUN_BASE_EVAL:
    base_metrics = evaluate_base_model(
        cfg=cfg,
        val_df=val_df,
        test_df=test_df,
        include_bertscore=True,
        logger=logger,
    )

    display(pd.DataFrame([base_metrics]))


In [11]:
# Optional: evaluate best adapter only (BERTScore included)
RUN_BEST_ADAPTER_EVAL = False

if RUN_BEST_ADAPTER_EVAL:
    best_adapter = Path(run_info["adapter_dir"])
    if not best_adapter.exists():
        raise FileNotFoundError(f"Best adapter not found: {best_adapter}")

    best_metrics = evaluate_single_adapter(
        cfg=cfg,
        adapter_dir=best_adapter,
        tag="best",
        val_df=val_df,
        test_df=test_df,
        include_bertscore=True,
        logger=logger,
    )

    display(pd.DataFrame([best_metrics]))


In [12]:
# Compare BASE (untuned) vs BEST CHECKPOINT SO FAR (tuned)
# using DeBERTa for BERTScore (no retraining)

import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import experiments.mistral.utils.pipeline as pipeline_mod
from experiments.utils.evaluation import (
    _get_bertscore,
    compute_eval_metrics as _orig_compute_eval_metrics,
)

BERTSCORE_MODEL_TYPE = "microsoft/deberta-xlarge-mnli"
BERTSCORE_BATCH_SIZE = 8
BERTSCORE_DEVICE = "cpu"  # safer; set to "cuda" only if GPU headroom is available
EVAL_GEN_BATCH_SIZE = 1  # reduce generation memory pressure


def _cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def _compute_eval_metrics_deberta(pred_df, include_bertscore=False):
    # Keep existing json/score/rouge/bleu behavior, replace only BERTScore settings.
    out = _orig_compute_eval_metrics(pred_df, include_bertscore=False)

    if include_bertscore:
        preds = pred_df["pred_rationale"].fillna("").astype(str).tolist()
        refs = pred_df["rationale"].fillna("").astype(str).tolist()
        bert = _get_bertscore().compute(
            predictions=preds,
            references=refs,
            model_type=BERTSCORE_MODEL_TYPE,
            batch_size=BERTSCORE_BATCH_SIZE,
            device=BERTSCORE_DEVICE,
        )
        out["bertscore_precision"] = float(np.mean(bert["precision"]))
        out["bertscore_recall"] = float(np.mean(bert["recall"]))
        out["bertscore_f1"] = float(np.mean(bert["f1"]))

    return out


RUN_COMPARE_BASE_VS_BEST = True

if RUN_COMPARE_BASE_VS_BEST:
    output_dir = cfg.output_root / "models" / cfg.run_name
    ckpts = sorted(
        [p for p in output_dir.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: int(p.name.split("-")[-1]),
    )
    if not ckpts:
        raise FileNotFoundError(f"No checkpoints found in: {output_dir}")

    latest_ckpt = ckpts[-1]
    trainer_state_path = latest_ckpt / "trainer_state.json"
    if not trainer_state_path.exists():
        raise FileNotFoundError(f"Missing trainer_state.json: {trainer_state_path}")

    state = json.loads(trainer_state_path.read_text(encoding="utf-8"))
    best_ckpt_str = state.get("best_model_checkpoint")
    if not best_ckpt_str:
        raise ValueError("best_model_checkpoint is missing in trainer_state.json")

    best_ckpt = Path(best_ckpt_str)
    if not best_ckpt.exists():
        raise FileNotFoundError(f"best_model_checkpoint path does not exist: {best_ckpt}")

    print("Latest checkpoint:", latest_ckpt)
    print("Best checkpoint:", best_ckpt)
    print("Best eval_loss:", state.get("best_metric"))
    print("BERTScore model:", BERTSCORE_MODEL_TYPE)
    print("BERTScore device:", BERTSCORE_DEVICE)

    prev_wandb = cfg.wandb.enabled
    prev_gen_bs = cfg.generation.batch_size
    old_compute = pipeline_mod.compute_eval_metrics

    cfg.wandb.enabled = False
    cfg.generation.batch_size = EVAL_GEN_BATCH_SIZE
    pipeline_mod.compute_eval_metrics = _compute_eval_metrics_deberta

    try:
        _cleanup_cuda()

        base_metrics = evaluate_base_model(
            cfg=cfg,
            val_df=val_df,
            test_df=test_df,
            include_bertscore=True,
            logger=logger,
        )

        _cleanup_cuda()

        best_metrics = evaluate_single_adapter(
            cfg=cfg,
            adapter_dir=best_ckpt,
            tag=f"best_ckpt_{best_ckpt.name}",
            val_df=val_df,
            test_df=test_df,
            include_bertscore=True,
            use_wandb=False,
            logger=logger,
        )
    except RuntimeError as e:
        if "illegal memory access" in str(e).lower():
            raise RuntimeError(
                "CUDA illegal memory access during eval. Restart the kernel, then rerun this cell. "
                "If it still fails, keep EVAL_GEN_BATCH_SIZE=1 and BERTSCORE_DEVICE='cpu'."
            ) from e
        raise
    finally:
        pipeline_mod.compute_eval_metrics = old_compute
        cfg.wandb.enabled = prev_wandb
        cfg.generation.batch_size = prev_gen_bs
        _cleanup_cuda()

    comparison_df = pd.DataFrame([base_metrics, best_metrics])
    display(comparison_df)



Latest checkpoint: /home/HaniWB/my_gpu_project/Abstract-Evaluator/experiments/artifacts/abstract_evaluator_mistral7b_sft/models/mistral7b_abstract_evaluator_lora_l4_modular/checkpoint-1910
Best checkpoint: /home/HaniWB/my_gpu_project/Abstract-Evaluator/experiments/artifacts/abstract_evaluator_mistral7b_sft/models/mistral7b_abstract_evaluator_lora_l4_modular/checkpoint-764
Best eval_loss: 0.7619926333427429
BERTScore model: microsoft/deberta-xlarge-mnli
BERTScore device: cpu
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/HaniWB/my_gpu_project/Abstract-Evaluator/.venv/lib/python3.12/site-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.8: Fast Mistral patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.151 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu118. CUDA: 8.0. CUDA Toolkit: 11.8. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

[accelerate.big_modeling|WARNING]Some parameters are on the meta device because they were offloaded to the cpu.
2026-05-26 20:44:24 | INFO | mistral7b_abstract_evaluator_lora_l4_modular_pipeline | Generating validation predictions for base model


RuntimeError: CUDA error: an illegal memory access was encountered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Compare best_adapter vs base_model predictions and save as JSONL
RUN_SAVE_COMPARISON_JSONL = True

if RUN_SAVE_COMPARISON_JSONL:
    best_pred_path = cfg.output_root / "eval" / cfg.run_name / "test_predictions_best.csv"
    base_pred_path = cfg.output_root / "eval" / f"{cfg.run_name}_base_no_finetune" / "test_predictions_base_no_finetune.csv"

    if not best_pred_path.exists():
        raise FileNotFoundError(f"Missing best-adapter predictions: {best_pred_path}")
    if not base_pred_path.exists():
        raise FileNotFoundError(f"Missing base-model predictions: {base_pred_path}")

    best_pred = pd.read_csv(best_pred_path)
    base_pred = pd.read_csv(base_pred_path)

    join_keys = [c for c in ["id", "paper_id"] if c in best_pred.columns and c in base_pred.columns]
    if not join_keys:
        join_keys = ["score", "rationale"]

    compare_df = best_pred.merge(
        base_pred,
        on=join_keys,
        how="inner",
        suffixes=("_best_adapter", "_base_model"),
    )

    compare_df["pred_score_diff_best_minus_base"] = (
        compare_df["pred_score_best_adapter"] - compare_df["pred_score_base_model"]
    )

    comparison_path = cfg.output_root / "eval" / cfg.run_name / "best_vs_base_test_comparison.jsonl"
    comparison_path.parent.mkdir(parents=True, exist_ok=True)
    compare_df.to_json(comparison_path, orient="records", lines=True, force_ascii=False)

    print({"comparison_rows": len(compare_df), "saved_to": str(comparison_path)})
    display(compare_df.head(10))


In [ ]:
# Separate generation step (model-agnostic utility)
# This demonstrates using generate_predictions_from_messages independently.
RUN_GENERATION_STEP_ONLY = False

if RUN_GENERATION_STEP_ONLY:
    from experiments.mistral.utils.chat import make_inference_prompt

    adapter_dir = Path(run_info["adapter_dir"])
    model, tokenizer = load_mistral_model_for_inference(
        model_name=cfg.model_name,
        adapter_dir=adapter_dir if adapter_dir.exists() else None,
        max_seq_length=cfg.max_seq_length,
    )

    demo_pred = generate_predictions_from_messages(
        eval_df=val_df.head(8),
        model=model,
        tokenizer=tokenizer,
        make_inference_prompt_fn=make_inference_prompt,
        parse_score_fn=parse_score,
        parse_rationale_fn=parse_rationale,
        max_seq_length=cfg.max_seq_length,
        max_new_tokens=cfg.generation.max_new_tokens,
        batch_size=2,
    )

    demo_pred.head()
